# Data — analyzer

**Block 3 of 3 in the Data stage, step 3 of 8.** The other two blocks *produce* data; this notebook
is where it is *understood*, and where a feature either earns a backtest or is dropped before
anybody spends a week on it.

**In plain words:** build features and test whether they carry signal, before you model anything.

**It produces** the charts and the information-coefficient table in `Data/Analyzer/`.

**It prevents** a book built on a feature that never predicted anything.

> **Runs after `Data/refinery.py`, and before any experiment** — the predictions an experiment's
> blueprint makes are supposed to come from here.

```
Data/curator.py    + Data/Curator/custom_calculations.py    ->  Curator/Time_Series/   m_* + c_*
Data/refinery.py   + Data/Refinery/custom_calculations.py   ->  Refinery/Time_Series/  + r_*
Data/analyzer.ipynb                                         ->  Analyzer/  charts + the IC table
```

`Universe/universe.ipynb` profiles the *catalogue* — what exists, what is missing, when each
security becomes usable. This notebook looks at the **content**: what the data says, and whether
the signal built on it carries anything.

**In the template this notebook holds no code:** each section below says what is expected in it.
Write the cells.

## 0 · Setup

Read the refined panel, and name **the columns this notebook reads, once, in this cell** — the
eligibility column, the features it eats, the cross-sectional candidates to screen. Everything
below re-runs unchanged when they change.

| Column family | Built by | Scope |
| --- | --- | --- |
| `m_*` | the provider, via the Curator | raw market data |
| `c_*` | `Curator/custom_calculations.py` | **per security** — one security's own history |
| `r_*` | `Refinery/custom_calculations.py` | **cross-sectional**, or with a setting an experiment sweeps |
| `current_*` | joined from the security master | today's classification, **not point-in-time** |

In [ ]:
# EXAMPLE-ONLY CELL
import itertools
import os
import pathlib
import sys

import numpy
import pandas

NOTEBOOK_DIRECTORY = pathlib.Path.cwd()
REPOSITORY_ROOT = (
    NOTEBOOK_DIRECTORY
    if (NOTEBOOK_DIRECTORY / "Experiments").is_dir()
    else NOTEBOOK_DIRECTORY.parent
)
os.chdir(REPOSITORY_ROOT)
sys.path.insert(0, str(REPOSITORY_ROOT / "Experiments"))

import securities_panel  # noqa: E402 - the path above has to exist first

# The columns this notebook reads, named once. Everything below re-runs unchanged when they change.
SIGNAL_COLUMN = "r_trend_50_200"
LIQUIDITY_COLUMN = "r_liquidity_rank"
MEMBERSHIP_COLUMN = "r_index_weight"
TRADED_VALUE_COLUMN = "r_traded_value_sma_63d"
CURATOR_TRADED_VALUE_COLUMN = "c_daily_traded_value_sma_63d"
PRICE_COLUMN = "m_close_dividend_and_split_adjusted"
# The rule ranks the golden-cross members by traded value and holds a few dozen, so the pool it
# actually chooses from is the most traded members, not the whole index.
POOL_SIZE = 50
HORIZONS_IN_DAYS = (21, 63, 252)
ANALYZER_DIRECTORY = pathlib.Path("Data/Analyzer")
# The window the universe notebook fixed, read from what it wrote, and the stretch before it, read
# apart: the experiment is judged on the first, and the second says whether what the first shows
# is older than the window.
# Everything after ANALYSIS_END is held out until a book is frozen: no measurement here has seen it.
window = pandas.read_csv("Universe/Window.csv")
WINDOW_START = pandas.Timestamp(window["window_start"].iloc[0])
ANALYSIS_END = pandas.Timestamp("2026-06-01")
BEFORE_START = pandas.Timestamp("2002-01-02")
BEFORE_END = WINDOW_START - pandas.Timedelta(1, unit="D")
PERIODS = (
    ("before the window", BEFORE_START, BEFORE_END),
    ("the window", WINDOW_START, ANALYSIS_END),
)

matrices = securities_panel.load_matrices((
    SIGNAL_COLUMN,
    LIQUIDITY_COLUMN,
    MEMBERSHIP_COLUMN,
    TRADED_VALUE_COLUMN,
    CURATOR_TRADED_VALUE_COLUMN,
    PRICE_COLUMN,
))
cut = {
    name: matrix.loc[:ANALYSIS_END]
    for name, matrix in matrices.items()
}
signal = cut[SIGNAL_COLUMN]
liquidity = cut[LIQUIDITY_COLUMN]
member = cut[MEMBERSHIP_COLUMN].fillna(0.0) > 0
traded_value = cut[TRADED_VALUE_COLUMN]
price = cut[PRICE_COLUMN]
print(f"panel: {price.shape[0]} dates x {price.shape[1]} securities")
print(f"from {price.index.min().date()} to {price.index.max().date()}")
print(f"members per date in the window: {int(member.loc[WINDOW_START:].sum(axis=1).min())} "
      f"to {int(member.loc[WINDOW_START:].sum(axis=1).max())}")

## 1 · What each stage contributed

The refined file is the Curator file plus columns, on its rows inside the span the seed gives the
security, or all of them where it gives none. Confirming that here is what lets every section
below read one folder and forget the Curator exists.

Then **coverage**, per column. A column at 60% coverage is not quietly averaged over the 60%: say
so, and decide whether the gap is a warm-up, a late listing or a broken input.

In [ ]:
# EXAMPLE-ONLY CELL
# Coverage per column, over the members only, so nothing below is averaged over non-members.
coverage = pandas.DataFrame({
    "populated": {
        name: int(matrix.where(member).notna().sum().sum())
        for name, matrix in cut.items()
    },
    "member_cells": {
        name: int(member.sum().sum())
        for name in cut
    },
})
coverage["share"] = (coverage["populated"] / coverage["member_cells"]).round(3)
print(coverage)

# The signal's gap among members is the 200-day warm-up of a name that listed late, not a broken
# input: it should be near zero from the window on.
signal_by_year = (
    signal.where(member).notna().sum(axis=1) / member.sum(axis=1)
).groupby(signal.index.year).mean().round(3)
print("share of members with a usable signal, by year:")
print(signal_by_year.loc[2002:].to_string())

## 2 · What diversification is actually available

A strategy that chooses between securities is a bet that they do not all move together. Whether
that is true is measurable, and it decides how much the strategy can possibly add: **if everything
is one trade, choosing between them is theatre.**

Buy-and-hold return, volatility and worst day per security; the correlation matrix; the mean
off-diagonal correlation and the extreme pairs.

In [ ]:
# EXAMPLE-ONLY CELL
# If everything is one trade, choosing between securities is theatre. This is the measurement, over
# the window, on the members. fill_method=None: a gap must stay a gap.
daily_returns = price.loc[WINDOW_START:].pct_change(fill_method=None)
ever_member = member.loc[WINDOW_START:].any(axis=0)
survivors = daily_returns.columns[ever_member & (daily_returns.notna().sum() > 500)]
sample = daily_returns[survivors]
correlations = sample.corr()
off_diagonal = ~numpy.eye(len(correlations), dtype=bool)
mean_correlation = numpy.nanmean(correlations.to_numpy()[off_diagonal])
print(f"members with more than 500 daily observations in the window: {len(survivors)}")
print(f"mean pairwise correlation: {mean_correlation:.3f}")

per_security = pandas.DataFrame({
    "annualised_return": (sample.mean() * 252).round(3),
    "annualised_volatility": (sample.std() * (252 ** 0.5)).round(3),
    "worst_day": sample.min().round(3),
})
print(per_security.describe().round(3))

## 3 · Are the cross-sectional columns what they claim to be?

The Refinery asserts one property that would be silent if broken: **every rank is a percentile
taken inside a single date.** A rank computed over the pooled sample would drift as the universe's
composition changed, and nothing would raise an error.

Check it as an identity, not as a rule of thumb. A per-date percentile over *n* untied values has
mean exactly `(n + 1) / (2n)` — 0.542 on twelve securities, 0.5006 on eight hundred. Testing
against 0.5 looks like a small failure on every date of a narrow universe and passes on a wide one,
which is the worst possible failure mode.

In [ ]:
# EXAMPLE-ONLY CELL
# A per-date percentile over n untied values averages to (n + 1) / (2n), not to 0.5. The rank is
# taken over the members only, so it should be null off them and satisfy the identity on them.
off_members = int(liquidity.where(~member).notna().sum().sum())
observed_mean = liquidity.mean(axis=1)
count_per_date = liquidity.notna().sum(axis=1)
expected_mean = (count_per_date + 1) / (2 * count_per_date)
difference = (observed_mean - expected_mean).abs()
tested = difference[count_per_date > 0]
print(f"ranks found on non-members: {off_members}")
print(f"dates tested: {len(tested)}")
print(f"largest deviation from the identity: {tested.max():.6f}")
print(f"dates off by more than 0.001: {int((tested > 0.001).sum())}")

# The rule ranks by the Refinery's 63-day average; it must be the Curator's to the last digit, so
# moving it to the Refinery changed no number.
curator_traded_value = cut[CURATOR_TRADED_VALUE_COLUMN]
both = traded_value.notna() & curator_traded_value.notna()
relative_gap = (
    (traded_value - curator_traded_value).abs() / curator_traded_value.abs().clip(lower=1.0)
).where(both)
largest_traded_value_gap = float(relative_gap.max().max())
print(f"largest relative gap between the two 63-day traded values: {largest_traded_value_gap:.2e}")

## 4 · Information coefficient — which features predict returns

**The section that decides things.** For each feature and horizon, the information coefficient is
the cross-sectional rank correlation between the feature on date *t* and the forward return from
*t* to *t+h*, computed **per date** and then averaged. Per date is what keeps it causal: the
correlation only ever compares securities observable at the same moment.

- **IC** — the mean daily correlation. The sign matters as much as the size: a negative IC means
  the feature works *inverted*.
- **IR** = IC / IC standard deviation — the consistency of the edge, which is what survives into a
  portfolio.
- **The share of dates with the expected sign**, beside the IR, and the overlap stated. A forward
  window of *h* days, measured every day, shares *h − 1* days with the next, so the dates are not
  independent observations and an IR counted over them overstates the evidence.

Compute it over the whole panel and, separately, over the **eligible pool** the strategy actually
selects from. The second is the one that matters: a feature can behave differently inside an
already-filtered group.

> **The IC table is a screening tool, not evidence.** An information ratio scales with the square
> root of the number of independent bets, so on a narrow universe read these as directional. **A
> feature that fails here does not get a book built on it**; one that passes has earned a backtest,
> not a belief.

<!-- example: begin -->

Here the panel is **the index's members on each date**, the pool **the fifty most traded of
them**, and both are read in two periods: **the window** the experiment is judged on, and
**before the window**, which says whether what the window shows is older than it.

<!-- example: end -->

In [ ]:
# EXAMPLE-ONLY CELL
# The rank correlation between the feature on date t and the forward return from t to t+h, taken
# per date over the members and then averaged. Per date is what keeps it causal.
traded_order = traded_value.where(member).rank(axis=1, ascending=False, method="first")
pool = traded_order <= POOL_SIZE
state = (signal > 0).astype(float).where(signal.notna())
rows = []

# Both halves of the idea are screened -- the trend the filter reads, as a distance and as the state
# the rule actually uses, and the traded-value rank the book is ranked and sized by -- each with
# the sign the claim expects: positive.
FEATURES = (
    (SIGNAL_COLUMN, signal),
    ("cross state, 1 above and 0 below", state),
    (LIQUIDITY_COLUMN, liquidity),
)
POOLS = (
    ("members", member),
    ("fifty most traded members", pool),
)

for horizon in HORIZONS_IN_DAYS:
    forward_return = price.shift(-horizon) / price - 1

    for (feature_name, feature_matrix), (pool_name, mask) in itertools.product(FEATURES, POOLS):
        daily_correlation = feature_matrix.where(mask).corrwith(
            forward_return.where(mask),
            axis=1,
            method="spearman",
        ).dropna()

        for period_name, period_start, period_end in PERIODS:
            in_period = daily_correlation.loc[period_start:period_end]
            information_coefficient = in_period.mean()
            rows.append({
                "feature": feature_name,
                "pool": pool_name,
                "period": period_name,
                "horizon_days": horizon,
                "overlap_days": horizon - 1,
                "information_coefficient": round(information_coefficient, 4),
                "information_ratio": round(information_coefficient / in_period.std(), 3),
                "share_expected_sign": round((in_period > 0).mean(), 3),
                "dates": len(in_period),
            })

information_table = pandas.DataFrame(rows)
ANALYZER_DIRECTORY.mkdir(parents=True, exist_ok=True)
information_table.to_csv(ANALYZER_DIRECTORY / "information_coefficient.csv", index=False)
print(information_table.to_string(index=False))

## 5 · The two questions any signal owes an answer to

Sections 0 to 4 are what any strategy needs. What belongs beside them depends on your signal, and
two are worth writing whatever it is.

**Does the signal separate anything?** Split forward return *and* forward volatility by the
signal's state. A signal can be worth trading on the second alone; it would not be the first.

**If the signal is fitted, what is look-ahead worth?** Read the same model causally and smoothed
and report the gap. It is the cheapest audit in the process and routinely the largest number in it:
the two series agree on most days and differ exactly at the turning points, which is where the
money is.

In [ ]:
# EXAMPLE-ONLY CELL
# Does the signal separate anything? Forward return AND forward volatility, split by the state the
# rule reads, inside the pool it selects from, over the window.
horizon = 21
forward_return = price.shift(-horizon) / price - 1
daily = price.pct_change(fill_method=None)
forward_volatility = daily.rolling(horizon).std().shift(-horizon) * (252 ** 0.5)
in_window = pandas.Series(price.index >= WINDOW_START, index=price.index)
window_pool = pool & pandas.DataFrame(
    numpy.repeat(in_window.to_numpy()[:, None], pool.shape[1], axis=1),
    index=pool.index,
    columns=pool.columns,
)
uptrend = (signal > 0) & window_pool
downtrend = (signal <= 0) & window_pool

separation = pandas.DataFrame({
    "forward_return_21d": {
        "50-day above 200-day": forward_return.where(uptrend).stack().mean(),
        "50-day below 200-day": forward_return.where(downtrend).stack().mean(),
    },
    "forward_volatility_21d": {
        "50-day above 200-day": forward_volatility.where(uptrend).stack().mean(),
        "50-day below 200-day": forward_volatility.where(downtrend).stack().mean(),
    },
    "observations": {
        "50-day above 200-day": int(uptrend.sum().sum()),
        "50-day below 200-day": int(downtrend.sum().sum()),
    },
})
separation.to_csv(ANALYZER_DIRECTORY / "separation.csv")
print(separation.round(4))

# How concentrated is traded value among the golden-cross members? A property of the signal and
# the market, never of a book: on each date, the largest member's share of the cross
# members' traded value, and how many members hold at least one percent of it.
cross_value = traded_value.where(member & (signal > 0))
cross_share = cross_value.div(cross_value.sum(axis=1), axis=0)
concentration = pandas.DataFrame({
    "eligible_members": cross_value.notna().sum(axis=1),
    "largest_share": cross_share.max(axis=1),
    "members_at_one_percent_or_more": (cross_share >= 0.01).sum(axis=1),
}).loc[WINDOW_START:]
concentration_by_year = concentration.groupby(concentration.index.year).median().round(3)
concentration_by_year.to_csv(ANALYZER_DIRECTORY / "concentration.csv")
print(concentration_by_year.to_string())

# This signal is arithmetic on past prices, not a fitted model, so there is no smoothed reading to
# compare a causal one against: the look-ahead audit costs nothing here and is worth zero.
print("signal is unfitted: causal and smoothed readings are the same series")

## 6 · Handoff

| Output | Consumed by |
| --- | --- |
| `Data/Refinery/Time_Series/` | **`Experiments/` — read this one** |
| `Data/Analyzer/` — the IC table | feature selection in every experiment |
| `Data/Analyzer/Charts/` | `FINDINGS_N.md` |

## What to write in the blueprint before running an experiment

**Write down what you expect before you run this.** A prediction made from the data and then
confirmed by the engine is the strongest methodological result an experiment can report; a number
found first and explained afterwards is a story.

Put the predictions this notebook licenses in `BLUEPRINT_N.md` **before** the backtest, each with
the section it came from and the observation that would falsify it. **Findings from this stage go
straight into `RESULTS.md`**, under *Before any experiment* — notebook outputs are stripped before
committing, so a measurement living only in a cell output does not survive the commit.

In [ ]:
# EXAMPLE-ONLY CELL
print(f"{ANALYZER_DIRECTORY / 'information_coefficient.csv'} -> BLUEPRINT_1.md predictions")
print("Findings from this stage go straight into RESULTS.md, under 'Before any experiment':")
print(information_table.to_string(index=False))
print(separation.round(4).to_string())
print(concentration_by_year.to_string())

## 7 · Verify

**Assertions that raise when this stage's output is wrong**, read back from what this notebook
wrote to `Data/Analyzer/` rather than from the variables that wrote it. A notebook that reaches its
last cell is then one whose table can be quoted. At the least:

- the coefficient table has a row for every feature, pool and horizon, each with a finite
  coefficient, ratio and share of dates with the expected sign;
- no coefficient is averaged over no dates, nor over more dates than the panel has a forward
  return for at its horizon;
- the pool the strategy selects from never counts more dates than the whole panel at the same
  horizon, because it is a part of it.

<!-- example: begin -->

Here the Verify section checks that the panel ends on or before 2026-06-01, so nothing has seen
the held-out months; that the coefficient table has a row for every feature, pool, period and
horizon, each finite, with a share of dates with the expected sign between 0 and 1; that no
row averages over no dates and the pool never counts more dates than the members; and that no
rank sits on a non-member and the rule's 63-day traded value is the Curator's. It does not
yet count each row's dates against the forward returns its horizon has, which the list above
asks for.

<!-- example: end -->

In [ ]:
# EXAMPLE-ONLY CELL
# Read back the tables this notebook wrote, and raise on the first thing that is wrong.
table_path = ANALYZER_DIRECTORY / "information_coefficient.csv"
written_table = pandas.read_csv(table_path)
measured = written_table[[
    "information_coefficient",
    "information_ratio",
    "share_expected_sign",
]]
dates_by_pool = written_table.pivot_table(
    index=["feature", "period", "horizon_days"],
    columns="pool",
    values="dates",
)
verifications = {
    "the panel ends no later than the held-out boundary": price.index.max() <= ANALYSIS_END,
    "a row for every feature, pool, period and horizon": (
        len(written_table) == len(FEATURES) * len(POOLS) * len(PERIODS) * len(HORIZONS_IN_DAYS)
    ),
    "every coefficient, ratio and sign share is finite": bool(
        numpy.isfinite(measured).all(axis=None)
    ),
    "every sign share lies between 0 and 1": bool(
        measured["share_expected_sign"].between(0, 1).all()
    ),
    "every row averages over at least one date": bool((written_table["dates"] > 0).all()),
    "the pool never counts more dates than the members": bool(
        (dates_by_pool["fifty most traded members"] <= dates_by_pool["members"]).all()
    ),
    "no rank sits on a non-member": off_members == 0,
    "the rule's traded value is the Curator's": largest_traded_value_gap < 1e-6,
    "the separation and concentration tables were written": (
        (ANALYZER_DIRECTORY / "separation.csv").is_file()
        and (ANALYZER_DIRECTORY / "concentration.csv").is_file()
    ),
}
failed = [
    name
    for name, passed in verifications.items()
    if not passed
]

if len(failed) > 0:
    message = f"the analyzer failed verification: {'; '.join(failed)}"

    raise AssertionError(message)

print(f"verified: {len(verifications)} checks on {table_path}")